# Normalize Morita et al. lipid names to GOSLIN format

**Goal:** Convert proprietary lipid names from Morita et al. into GOSLIN-normalized form and validate them against the GOSLIN REST API.

**Steps:**
1. Load raw lipidomics data (`lipid_Morita_et_al.xlsx`)
2. Normalize lipid names to GOSLIN-compatible format (regex-based pre-processing)
3. Validate normalized names via GOSLIN REST API and retrieve hierarchy metadata
4. Calculate and report match rate
5. Save results to `data/processed/goslin_Morita.csv`

In [5]:
import os
import re
import time
import requests
import pandas as pd
from urllib.parse import quote


## Step 1 — Load raw lipidomics data

In [6]:
# Load unique lipid names from the main sheet of Morita et al.
df_raw = pd.read_excel('../data/raw/lipid_Morita_et_al.xlsx', sheet_name='Lipid_all')

lipid_names = df_raw['CompoundName'].dropna().unique().tolist()

print(f"Total unique lipid names: {len(lipid_names)}")
lipid_names[:5]

Total unique lipid names: 1306


['Cer d18:1_14:0',
 'Cer d18:1_16:0',
 'Cer d18:1_16:1',
 'Cer d18:1_18:0',
 'Cer d18:1_18:1']

## Step 2 — Normalize lipid names to GOSLIN-compatible format

Morita et al. uses a proprietary naming convention that differs from GOSLIN in three ways:

| Pattern | Morita format | GOSLIN format |
|---|---|---|
| Multi-chain with parentheses | `TG(12:0)(14:1)(18:2)` | `TG 12:0/14:1/18:2` |
| Ceramide with `d` prefix | `Cer d18:1_16:0` | `Cer 18:1;2/16:0` |
| Underscore chain separator | `PC 16:0_18:1` | `PC 16:0/18:1` |

In [7]:
def normalize_lipid_name(name):
    """
    Convert Morita et al. proprietary lipid names to GOSLIN-compatible format.

    Rules applied in order:
    1. Parenthesis chains -> underscore-separated (molecular species: sn-position unknown)
       e.g. TG(12:0)(14:1)(18:2) -> TG 12:0_14:1_18:2
    2. Ceramide 'd' prefix -> ';2' hydroxyl notation, preserving '_'
       e.g. Cer d18:1_16:0 -> Cer 18:1;2_16:0

    NOTE: '_' and '/' are NOT interchangeable.
    '_' = molecular species (sn-position unknown, LIPID MAPS Shorthand)
    '/' = structural subspecies (sn-position known)
    Morita et al. use '_' throughout, so '_' is preserved throughout normalization.
    Rule 3 (underscore -> slash) has been removed as it fabricated sn-position data.
    """
    # Rule 1: parenthesis-enclosed chains (handles 1-, 2-, 3-chain lipids)
    # Use '_' (not '/') because Morita's parenthesis notation does not specify sn-positions.
    m = re.match(r'^([A-Za-z][A-Za-z0-9]*)((?:\([^)]+\))+)$', name)
    if m:
        cls    = m.group(1)
        chains = re.findall(r'\(([^)]+)\)', m.group(2))
        name   = cls + ' ' + '_'.join(chains)

    # Rule 2: Ceramide 'd' prefix (dihydroxy sphingoid base)
    # Use '_' to preserve the sn-position-unknown status of the Morita data.
    # GOSLIN API will return '/' in its normalized form; preserve_sn_ambiguity restores '_'.
    name = re.sub(r'((?:Hex)?Cer) d(\d+:\d+)_(\S+)', r'\1 \2;2_\3', name)

    return name


def preserve_sn_ambiguity(original_name: str, converted_name) -> str:
    """
    Post-process GOSLIN API output to restore '_' where the original had '_'.

    GOSLIN always serializes chain separators as '/' in its normalized output.
    When the source data used '_' (sn-position unknown), we must restore '_'
    to avoid fabricating positional information that is absent from the raw data.
    Also handles NaN (float) values that arise when loading normalized_name from CSV.
    """
    if not isinstance(converted_name, str):
        return converted_name  # None or NaN from CSV
    if '_' in original_name:
        converted_name = converted_name.replace('/', '_')
    return converted_name


# Apply normalization to all lipid names
lipid_names_normalized = [normalize_lipid_name(n) for n in lipid_names]

# Spot-check conversion examples
examples = [
    'TG(12:0)(14:1)(18:2)',
    'PC(16:0)(18:1)',
    'MG(18:1)',
    'Cer d18:1_14:0',
    'HexCer d18:1_24:1',
    'PC 16:0_18:1',
    'Cholesterol',
]
print("Normalization examples (all outputs must use '_', never '/' from '_' input):")
for n in examples:
    result = normalize_lipid_name(n)
    flag = ' *** FABRICATED /' if ('_' in n and '/' in result) else ''
    print(f"  {n:35s} -> {result}{flag}")

# Verify: no '_'-containing input maps to '/'-containing intermediate
violations = [
    (n, r) for n, r in zip(lipid_names, lipid_names_normalized)
    if '_' in n and '/' in r
]
assert len(violations) == 0, (
    f"{len(violations)} names had '_' in original but '/' in intermediate:\n"
    + '\n'.join(f"  {o} -> {r}" for o, r in violations[:5])
)
print(f"\nVerification passed: 0 of {len(lipid_names)} names fabricate '/' from '_' input")


Normalization examples (all outputs must use '_', never '/' from '_' input):
  TG(12:0)(14:1)(18:2)                -> TG 12:0_14:1_18:2
  PC(16:0)(18:1)                      -> PC 16:0_18:1
  MG(18:1)                            -> MG 18:1
  Cer d18:1_14:0                      -> Cer 18:1;2_14:0
  HexCer d18:1_24:1                   -> HexCer 18:1;2_24:1
  PC 16:0_18:1                        -> PC 16:0_18:1
  Cholesterol                         -> Cholesterol

Verification passed: 0 of 1306 names fabricate '/' from '_' input


## Step 3 — Validate via GOSLIN REST API and retrieve hierarchy metadata

Send each normalized name to the GOSLIN REST API. On success, retrieve:
- `normalized_name`: GOSLIN canonical form
- `lipid_level`: granularity level (SPECIES, MOLECULAR_SPECIES, SN_POSITION, …)
- `category` / `class` / `class_name`: LipidMaps classification
- `mass`, `formula`: physicochemical properties

Failed names (class-only names without chain info, e.g. `"PC"`, `"TG"`) are collected separately.

> **Note:** The API has a rate limit; `time.sleep(0.1)` is applied between requests.

In [8]:
CACHE_PATH = '../data/processed/goslin_Morita.csv'
GOSLIN_API = "https://metabocloud.mesocentre.uca.fr/goslin/validate"

if os.path.exists(CACHE_PATH):
    # ── Cache hit: load from CSV, skip API calls ──────────────────────────────
    print(f"Cache found: loading from '{CACHE_PATH}' (delete to re-fetch)")
    df_goslin = pd.read_csv(CACHE_PATH)

    # Reconstruct results from the cached DataFrame.
    # Re-apply preserve_sn_ambiguity in case the cache was built with old code
    # that serialised '/' where the original had '_'.
    results = {}
    for _, row in df_goslin.iterrows():
        orig = row['original_name']
        d    = row.drop('original_name').to_dict()
        d['normalized_name'] = preserve_sn_ambiguity(orig, d.get('normalized_name'))
        results[orig] = d

    matched_originals = set(df_goslin['original_name'])
    failed = [n for n in lipid_names if n not in matched_originals]

else:
    # ── Cache miss: call GOSLIN REST API (runs once, then cached) ─────────────
    print("No cache found — calling GOSLIN REST API (this may take a few minutes)...")

    results = {}
    failed  = []

    for original, normalized in zip(lipid_names, lipid_names_normalized):
        try:
            response = requests.get(f"{GOSLIN_API}?lipid_names={quote(normalized)}")
            data     = response.json()

            if data['nb_success'] > 0:
                lipid = data['lipid_list'][0]
                results[original] = {
                    # preserve_sn_ambiguity restores '_' where GOSLIN returned '/'
                    # for names where the original Morita data had '_' (sn-position unknown).
                    'normalized_name': preserve_sn_ambiguity(original, lipid.get('normalized_name')),
                    'lipid_level':     lipid.get('lipid_level'),
                    'category':        lipid.get('lipidmaps_category'),
                    'class':           lipid.get('lipidmaps_class'),
                    'class_name':      lipid.get('class_name'),
                    'extended_class':  lipid.get('extended_class'),
                    'mass':            lipid.get('mass'),
                    'formula':         lipid.get('formula'),
                }
            else:
                failed.append(original)

            time.sleep(0.1)

        except Exception as e:
            print(f"Error for '{original}': {e}")
            failed.append(original)

    print(f"Done — API returned {len(results)} successes, {len(failed)} failures")


Cache found: loading from '../data/processed/goslin_Morita.csv' (delete to re-fetch)


## Step 4 — Match rate and results

In [12]:
n_total   = len(lipid_names)
n_matched = len(results)
n_failed  = len(failed)

print(f"Total lipid names  : {n_total}")
print(f"Matched (success)  : {n_matched}  ({n_matched / n_total * 100:.1f}%)")
print(f"Unmatched (failed) : {n_failed}   ({n_failed  / n_total * 100:.1f}%)")
print()
print("Unmatched names (typically class-level names without chain info):")
for name in failed:
    print(f"  {name}")

# Build result DataFrame
df_goslin = (
    pd.DataFrame.from_dict(results, orient='index')
    .rename_axis('original_name')
    .reset_index()
)[['original_name', 'normalized_name', 'lipid_level',
   'category', 'class', 'class_name', 'extended_class', 'mass', 'formula']]

print(f"\nResult shape: {df_goslin.shape}")
df_goslin[20:40]

Total lipid names  : 1306
Matched (success)  : 1290  (98.8%)
Unmatched (failed) : 16   (1.2%)

Unmatched names (typically class-level names without chain info):
  Cer
  DG
  LPC
  LPE
  MG
  PA
  PC
  PE
  PG
  PI
  PS
  HexCer
  SM
  CE
  FA
  TG

Result shape: (1290, 9)


,original_name,normalized_name,lipid_level,category,class,class_name,extended_class,mass,formula
20,Cer d18:1_22:6,NaN,SN_POSITION,SP,Ceramides [SP02],Cer,Cer,609.512095,C40H67NO3
21,Cer d18:1_24:0,NaN,SN_POSITION,SP,Ceramides [SP02],Cer,Cer,649.637296,C42H83NO3
22,Cer d18:1_24:1,NaN,SN_POSITION,SP,Ceramides [SP02],Cer,Cer,647.621646,C42H81NO3
23,Cer d18:1_26:0,NaN,SN_POSITION,SP,Ceramides [SP02],Cer,Cer,677.668596,C44H87NO3
24,Cholesterol,NaN,FULL_STRUCTURE,ST,Cholesterol and derivatives [ST0101],Cholesterol,ST 27:1;O,386.354866,C27H46O
25,DG 12:0_16:0,NaN,STRUCTURE_DEFINED,GL,Diacylglycerols [GL0201],DG,DG,512.444075,C31H60O5
26,DG 12:0_16:1,NaN,SN_POSITION,GL,Diacylglycerols [GL0201],DG,DG,510.428425,C31H58O5
27,DG 12:0_18:0,NaN,STRUCTURE_DEFINED,GL,Diacylglycerols [GL0201],DG,DG,540.475375,C33H64O5
28,DG 12:0_18:1,NaN,SN_POSITION,GL,Diacylglycerols [GL0201],DG,DG,538.459725,C33H62O5
29,DG 12:0_18:2,NaN,SN_POSITION,GL,Diacylglycerols [GL0201],DG,DG,536.444075,C33H60O5


In [10]:
# GOSLIN lipid_level distribution
# Finest to coarsest: FULL_STRUCTURE > COMPLETE_STRUCTURE > SN_POSITION >
#                     STRUCTURE_DEFINED > MOLECULAR_SPECIES > SPECIES > CLASS
GOSLIN_LEVEL_ORDER = [
    'FULL_STRUCTURE',
    'COMPLETE_STRUCTURE',
    'SN_POSITION',
    'STRUCTURE_DEFINED',
    'MOLECULAR_SPECIES',
    'SPECIES',
    'CLASS',
    'UNDEFINED',
]

level_counts = df_goslin['lipid_level'].value_counts(dropna=False)
n_total = len(df_goslin)

rows = []
for lv in GOSLIN_LEVEL_ORDER:
    n = int(level_counts.get(lv, 0))
    if n > 0:
        rows.append({'lipid_level': lv, 'n': n, 'pct': f'{n / n_total * 100:.1f}%'})

n_other = int(df_goslin['lipid_level'].isna().sum())
if n_other > 0:
    rows.append({'lipid_level': '(unmatched)', 'n': n_other, 'pct': f'{n_other / n_total * 100:.1f}%'})

df_level_summary = pd.DataFrame(rows)
print(f'GOSLIN lipid_level summary (n={n_total}):')
print()
print(df_level_summary.to_string(index=False))

GOSLIN lipid_level summary (n=1290):

       lipid_level    n   pct
    FULL_STRUCTURE    1  0.1%
COMPLETE_STRUCTURE   36  2.8%
       SN_POSITION 1172 90.9%
 STRUCTURE_DEFINED   12  0.9%
 MOLECULAR_SPECIES   43  3.3%
           SPECIES   26  2.0%


In [11]:
# Save to CSV (avoids re-running the API calls in downstream notebooks)
out_path = '../data/processed/goslin_Morita.csv'
df_goslin.to_csv(out_path, index=False)
print(f"Saved: {out_path}")

Saved: ../data/processed/goslin_Morita.csv
